# 02 — Clean & Standardize

Load the raw chart into DuckDB and clean it **in SQL** (workspace norm):
strip `$`/commas, cast to numeric, derive `decade` and the `inflation_multiple`
(adjusted / nominal). Result: the `films_adjusted` table.

In [ ]:
import sys, os
from pathlib import Path
PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

import pandas as pd
from io import StringIO
from src.ingest import load_config
from src.clean_quality import get_connection, load_to_duckdb, run_sql, register_source, save_interim

cfg = load_config('config.yaml')
con = get_connection(cfg)

# Reload the raw HTML parsed in 01 into a raw DuckDB table.
html = (Path(cfg['paths']['data_raw']) / 'bom_top_lifetime_adjusted_2022.html').read_text(encoding='utf-8')
raw = pd.read_html(StringIO(html))[0]
load_to_duckdb(raw, 'bom_raw', con)
print('bom_raw:', con.execute('SELECT COUNT(*) FROM bom_raw').fetchone()[0], 'rows')

## Clean in DuckDB
Money strings (`$1,895,421,694`) become `BIGINT`; `decade` uses integer
division; `inflation_multiple` = adjusted / nominal (how many times the film's
original take the adjusted figure represents).

In [ ]:
films = run_sql('''
    SELECT
        CAST("Rank" AS INTEGER)                                   AS rank_adjusted,
        "Title"                                                   AS title,
        CAST(REGEXP_REPLACE("Adj. Lifetime Gross", '[$,]', '', 'g') AS BIGINT) AS adjusted_gross,
        CAST(REGEXP_REPLACE("Lifetime Gross",       '[$,]', '', 'g') AS BIGINT) AS nominal_gross,
        CAST("Est. Num Tickets" AS BIGINT)                        AS est_tickets,
        CAST("Year" AS INTEGER)                                   AS release_year,
        (CAST("Year" AS INTEGER) // 10) * 10                      AS decade,
        ROUND(
            CAST(REGEXP_REPLACE("Adj. Lifetime Gross", '[$,]', '', 'g') AS DOUBLE)
            / NULLIF(CAST(REGEXP_REPLACE("Lifetime Gross", '[$,]', '', 'g') AS DOUBLE), 0),
            2) AS inflation_multiple
    FROM bom_raw
    ORDER BY adjusted_gross DESC
''', con)
load_to_duckdb(films, 'films_adjusted', con)
print(films.shape)
films.head(10)

## Quick sanity checks

In [ ]:
# The classics should dominate the adjusted board; recent blockbusters have
# the smallest inflation multiples (little time for ticket prices to rise).
print('Top adjusted:', films.iloc[0]['title'], films.iloc[0]['release_year'])
print('No nulls in key cols:',
      films[['adjusted_gross','nominal_gross','est_tickets','release_year']].notna().all().all())
films[['title','release_year','inflation_multiple']].sort_values('inflation_multiple').head(5)

## Save interim + register provenance

In [ ]:
save_interim(films, cfg, 'films_adjusted.parquet')

register_source(
    con, 'films_adjusted',
    name='Box Office Mojo - Top Lifetime Adjusted Grosses (domestic)',
    url=cfg['sources']['bom_adjusted']['url'],
    license='Data (c) IMDb/Box Office Mojo; used for commentary/analysis.',
    notes='Domestic (US/Canada) lifetime grosses. Adjusted gross in 2022 dollars via estimated tickets sold x 2022 average ticket price (ticket-price inflation, NOT CPI).',
    methodology='BOM estimates tickets sold x a reference-year average ticket price. Re-releases are included in a film lifetime total.',
    series_breaks='Nominal grosses are in year-of-release dollars and are NOT comparable across eras without the adjustment.',
)
print(con.execute('SELECT duckdb_table, source_name FROM _sources').df().to_string(index=False))

---
**Next:** `03-prepare.ipynb` packages the sellable export + codebook.

## Cleanup
Close the DuckDB connection so the write lock is released.

In [ ]:
con.close()
print('connection closed')